# Sprint 7 · Webinar 23 · Data Analytics teórico (Valores atípicos, segmentación y Git/GitHub) · Student Version

**Programa:** Data Analytics  
**Sprint:** 7  
**Modalidad:** Teórica guiada con demostraciones en Python  
**Duración sugerida:** 100 minutos

> Usa este cuaderno como material de clase. La idea no es solo ejecutar: también debes **tomar apuntes**, **hacer predicciones**, **completar código** y **registrar conclusiones**.

## Fecha

Completa la información de la sesión:

- **Fecha:** `____ / ____ / 2026`
- **Instructor/a:** `____________________________________`
- **Grupo / cohorte:** `____________________________________`
- **Tema central:** `____________________________________`

## Objetivos de la sesión teórica

Al finalizar esta sesión, deberías poder:

1. Identificar **valores atípicos (outliers)** usando reglas estadísticas como **IQR** y **Z-score**.
2. Explicar por qué un outlier no siempre significa “error”.
3. Proponer estrategias para tratar outliers según el **contexto de negocio**.
4. Crear una segmentación sencilla con reglas de negocio usando `if`, `elif`, `else` y `apply()`.
5. Redactar un **Statistical Summary** breve y orientado a stakeholders.
6. Describir por qué **Git/GitHub** es útil para reproducibilidad y colaboración en analítica.

## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Contenido | Lo entendí / dudas |
|---:|---|---|---|
| 0–10 min | Calentamiento | Qué es un outlier y por qué importa | `__` |
| 10–25 min | Dataset | Crear y explorar un dataset sintético | `__` |
| 25–45 min | 7.3.1 | Detectar outliers con IQR y Z-score | `__` |
| 45–55 min | 7.3.2 | Decidir qué hacer con outliers | `__` |
| 55–70 min | 7.3.3 | Segmentación con `if/elif/else` | `__` |
| 70–80 min | 7.3.4 | Segmentación con `apply()` y múltiples salidas | `__` |
| 80–88 min | 7.3.5 | Statistical Summary | `__` |
| 88–100 min | 7.4 | Git/GitHub para analistas | `__` |

## Ejercicio 0 · Calentamiento en breakout rooms (discusión conceptual)

**Objetivo:** activar ideas previas sobre outliers y decisiones analíticas.

Discutan en grupo y registren sus ideas:

1. ¿Qué variable de negocio podría tener outliers?  
   `______________________________________________________________________`
2. Da un ejemplo donde un outlier sea un **error de datos**.  
   `______________________________________________________________________`
3. Da un ejemplo donde un outlier sea un **evento real pero raro**.  
   `______________________________________________________________________`
4. ¿Qué riesgo existe si eliminas outliers sin revisar el contexto?  
   `______________________________________________________________________`

**Conclusión del equipo (3–5 bullets):**

- `______________________________________________________________`
- `______________________________________________________________`
- `______________________________________________________________`
- `______________________________________________________________`

## Ejercicio 1 · Crear y explorar un dataset extenso para análisis

En este bloque trabajaremos con un dataset **sintético** que simula un caso típico de analítica de clientes:

- atributos demográficos y de adquisición,
- transacciones con montos y descuentos,
- métricas agregadas por cliente para análisis de outliers y segmentación.

### Antes de ejecutar
Completa primero:

- ¿Qué columnas esperarías encontrar en una tabla de clientes?  
  `______________________________________________________________________`
- ¿Qué métricas serían útiles para medir valor del cliente?  
  `______________________________________________________________________`
- ¿Qué variable crees que podría tener cola larga o valores extremos?  
  `______________________________________________________________________`

In [ ]:

# 1) Setup del entorno
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 50)

# Reproducibilidad
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:

# 2) Crear dataset sintético de clientes
# Revisa la lógica y anota qué parte del proceso entiendes mejor.

n_customers = 5000
customer_id = np.arange(1, n_customers + 1)

regions = ['Norte', 'Centro', 'Sur', 'Occidente', 'Oriente']
channels = ['Organic', 'Ads', 'Referral', 'Partners', 'Email']
plans = ['Basic', 'Standard', 'Premium']

df_customers = pd.DataFrame({
    'customer_id': customer_id,
    'region': rng.choice(regions, size=n_customers, p=[0.18, 0.25, 0.22, 0.18, 0.17]),
    'acquisition_channel': rng.choice(channels, size=n_customers, p=[0.35, 0.25, 0.15, 0.10, 0.15]),
    'plan': rng.choice(plans, size=n_customers, p=[0.55, 0.35, 0.10]),
})

# TODO:
# 1. Genera edades usando una distribución normal.
# 2. Limita las edades a un rango razonable.
# 3. Inserta algunos valores extremos para analizarlos después.
age = rng.normal(loc=38, scale=12, size=n_customers).round().astype(int)
age = np.clip(age, 16, 85)

outlier_idx = rng.choice(n_customers, size=10, replace=False)
age[outlier_idx[:5]] = rng.integers(1, 10, size=5)
age[outlier_idx[5:]] = rng.integers(95, 120, size=5)

df_customers["age"] = age

start_date = pd.Timestamp("2023-01-01")
end_date = pd.Timestamp("2025-12-01")
signup_days = rng.integers(0, (end_date - start_date).days, size=n_customers)
df_customers["signup_date"] = start_date + pd.to_timedelta(signup_days, unit="D")

df_customers.head()

In [ ]:

# 3) Crear dataset sintético de transacciones
# Completa o modifica los bloques marcados con TODO si quieres experimentar.

n_transactions = 120_000
transaction_id = np.arange(1, n_transactions + 1)
customer_for_tx = rng.choice(df_customers["customer_id"], size=n_transactions, replace=True)

categories = ['Suscripción', 'Add-on', 'Soporte', 'Capacitación', 'Servicios Profesionales']
payment_methods = ['Tarjeta', 'Transferencia', 'PSE', 'Efectivo', 'Billetera']

# Cola larga en montos
amount = rng.lognormal(mean=3.0, sigma=0.8, size=n_transactions)

plan_map = df_customers.set_index("customer_id")["plan"].to_dict()
plan_factor = np.array([{'Basic': 0.9, 'Standard': 1.0, 'Premium': 1.3}[plan_map[c]] for c in customer_for_tx])
amount = amount * plan_factor

discount_pct = rng.choice(
    [0, 5, 10, 15, 20, 30, 40, 50],
    size=n_transactions,
    p=[0.35, 0.12, 0.18, 0.13, 0.10, 0.07, 0.03, 0.02]
)

tx_start = pd.Timestamp("2024-01-01")
tx_end = pd.Timestamp("2025-12-31")
tx_days = rng.integers(0, (tx_end - tx_start).days + 1, size=n_transactions)
tx_date = tx_start + pd.to_timedelta(tx_days, unit="D")

df_transactions = pd.DataFrame({
    "transaction_id": transaction_id,
    "customer_id": customer_for_tx,
    "transaction_date": tx_date,
    "category": rng.choice(categories, size=n_transactions, p=[0.55, 0.15, 0.15, 0.08, 0.07]),
    "payment_method": rng.choice(payment_methods, size=n_transactions, p=[0.55, 0.18, 0.15, 0.05, 0.07]),
    "amount_gross": amount.round(2),
    "discount_pct": discount_pct
})

# TODO: calcula amount_net a partir de amount_gross y discount_pct
df_transactions["amount_net"] = (
    df_transactions["amount_gross"] * (1 - df_transactions["discount_pct"] / 100)
).round(2)

# Insertar algunos outliers altos
high_outlier_rows = rng.choice(n_transactions, size=25, replace=False)
df_transactions.loc[high_outlier_rows, "amount_gross"] = rng.uniform(20_000, 120_000, size=25).round(2)
df_transactions.loc[high_outlier_rows, "amount_net"] = (
    df_transactions.loc[high_outlier_rows, "amount_gross"] *
    (1 - df_transactions.loc[high_outlier_rows, "discount_pct"] / 100)
).round(2)

# Insertar reembolsos (montos negativos)
refund_rows = rng.choice(n_transactions, size=60, replace=False)
df_transactions.loc[refund_rows, "amount_gross"] = -rng.uniform(10, 2000, size=60).round(2)
df_transactions.loc[refund_rows, "amount_net"] = df_transactions.loc[refund_rows, "amount_gross"]

df_transactions.head()

In [ ]:

# 4) Crear métricas por cliente
# Pasa de eventos (transacciones) a métricas agregadas por cliente.

analysis_date = pd.Timestamp("2026-01-01")
tx = df_transactions.copy()

# TODO:
# 1. Crea una bandera is_refund.
# 2. Agrupa por customer_id.
# 3. Calcula métricas como num_transactions, num_refunds, total_spend, avg_ticket, max_ticket y last_tx_date.
tx["is_refund"] = tx["amount_net"] < 0

agg = tx.groupby("customer_id").agg(
    num_transactions=("transaction_id", "count"),
    num_refunds=("is_refund", "sum"),
    total_spend=("amount_net", "sum"),
    avg_ticket=("amount_net", "mean"),
    max_ticket=("amount_net", "max"),
    last_tx_date=("transaction_date", "max")
).reset_index()

agg["recency_days"] = (analysis_date - agg["last_tx_date"]).dt.days
agg["refund_rate"] = (agg["num_refunds"] / agg["num_transactions"]).round(4)

df_customer_metrics = df_customers.merge(agg, on="customer_id", how="left")

# Completa nulos para clientes sin compras
df_customer_metrics["num_transactions"] = df_customer_metrics["num_transactions"].fillna(0).astype(int)
df_customer_metrics["num_refunds"] = df_customer_metrics["num_refunds"].fillna(0).astype(int)
df_customer_metrics["total_spend"] = df_customer_metrics["total_spend"].fillna(0.0)
df_customer_metrics["avg_ticket"] = df_customer_metrics["avg_ticket"].fillna(0.0)
df_customer_metrics["max_ticket"] = df_customer_metrics["max_ticket"].fillna(0.0)
df_customer_metrics["last_tx_date"] = df_customer_metrics["last_tx_date"].fillna(pd.NaT)
df_customer_metrics["recency_days"] = df_customer_metrics["recency_days"].fillna(
    (analysis_date - df_customer_metrics["signup_date"]).dt.days
).astype(int)
df_customer_metrics["refund_rate"] = df_customer_metrics["refund_rate"].fillna(0.0)

df_customer_metrics.head()

In [ ]:

# 5) Exploración inicial del dataset
# TODO:
# - revisa la estructura general
# - identifica tipos de datos
# - observa si hay columnas que ya parezcan sospechosas

df_customer_metrics.info()

In [ ]:

# TODO:
# Genera un resumen estadístico para columnas numéricas clave.
# Luego responde:
# - ¿qué variable parece más sesgada?
# - ¿dónde sospechas que puede haber outliers?

num_cols = [
    "age", "num_transactions", "total_spend",
    "avg_ticket", "max_ticket", "recency_days", "refund_rate"
]

df_customer_metrics[num_cols].describe().T

### Registro de hallazgos del Ejercicio 1

Completa después de ejecutar:

- **Número de clientes:** `____________________________`
- **Número de columnas en df_customer_metrics:** `____________________________`
- **Variable con mayor dispersión aparente:** `____________________________`
- **Variable donde sospecho cola larga:** `____________________________`
- **¿Existen clientes sin transacciones?:** `____________________________`
- **Interpretación rápida de `recency_days`:**  
  `______________________________________________________________________`

## 7.3 · Análisis de valores atípicos y segmentación

### 7.3.1 Identificando valores atípicos con reglas estadísticas

Un **outlier** es una observación que se aleja del patrón general de los datos.

Completa con tus palabras:

- **IQR** sirve para  
  `______________________________________________________________________`
- Un valor se considera atípico con IQR cuando está fuera de  
  `______________________________________________________________________`
- **Z-score** mide  
  `______________________________________________________________________`
- ¿Cuál método parece más robusto para distribuciones sesgadas?  
  `______________________________________________________________________`

In [ ]:

# 7.3.1 A) Detección de outliers con IQR
# Completa la lógica y luego registra tus conclusiones.

metric = "total_spend"

# TODO: calcula Q1, Q3 e IQR
q1 = df_customer_metrics[metric].quantile(0.25)
q3 = df_customer_metrics[metric].quantile(0.75)
iqr = q3 - q1

# TODO: define límites inferior y superior
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_customer_metrics["outlier_iqr_total_spend"] = (
    (df_customer_metrics[metric] < lower_bound) |
    (df_customer_metrics[metric] > upper_bound)
)

outlier_count = df_customer_metrics["outlier_iqr_total_spend"].sum()
outlier_pct = outlier_count / len(df_customer_metrics)

print(f"IQR bounds para {metric}: [{lower_bound:,.2f}, {upper_bound:,.2f}]")
print(f"Outliers detectados (IQR): {outlier_count:,} ({outlier_pct:.2%})")

df_customer_metrics.loc[
    df_customer_metrics["outlier_iqr_total_spend"],
    ["customer_id", "plan", "region", "num_transactions", "total_spend", "max_ticket"]
].head(10)

In [ ]:

# 7.3.1 B) Detección de outliers con Z-score
# TODO:
# 1. Calcula media y desviación estándar.
# 2. Crea la columna z_total_spend.
# 3. Marca outliers cuando |z| > 3.

metric = "total_spend"
mean = df_customer_metrics[metric].mean()
std = df_customer_metrics[metric].std(ddof=0)

if std == 0:
    df_customer_metrics["z_total_spend"] = 0.0
else:
    df_customer_metrics["z_total_spend"] = (df_customer_metrics[metric] - mean) / std

df_customer_metrics["outlier_z_total_spend"] = df_customer_metrics["z_total_spend"].abs() > 3

outlier_count_z = df_customer_metrics["outlier_z_total_spend"].sum()
outlier_pct_z = outlier_count_z / len(df_customer_metrics)

print(f"Outliers detectados (Z-score |z|>3): {outlier_count_z:,} ({outlier_pct_z:.2%})")

df_customer_metrics.loc[
    df_customer_metrics["outlier_z_total_spend"],
    ["customer_id", "plan", "num_transactions", "total_spend", "z_total_spend"]
].head(10)

In [ ]:

# 7.3.1 C) Visualización rápida
# TODO:
# 1. Construye un histograma de total_spend.
# 2. Construye un boxplot.
# 3. Anota qué te muestra cada gráfico.

plt.figure(figsize=(10, 4))
plt.hist(df_customer_metrics["total_spend"], bins=60)
plt.title("Distribución de total_spend")
plt.xlabel("total_spend")
plt.ylabel("frecuencia")
plt.show()

plt.figure(figsize=(10, 2.5))
plt.boxplot(df_customer_metrics["total_spend"], vert=False)
plt.title("Boxplot de total_spend")
plt.xlabel("total_spend")
plt.show()

### Registro del análisis de outliers

Completa durante la clase:

- **Outliers detectados con IQR:** `____________________________`
- **Outliers detectados con Z-score:** `____________________________`
- **Método que detectó más casos:** `____________________________`
- **¿Los mismos clientes aparecen en ambos métodos?:**  
  `______________________________________________________________________`
- **¿Qué muestra el histograma que no muestra tan bien el boxplot?:**  
  `______________________________________________________________________`
- **Conclusión inicial:**  
  `______________________________________________________________________`

### Mini-ejercicio

Repite el análisis IQR para `max_ticket` o para otra variable numérica.

Completa:

- **Variable elegida:** `____________________________`
- **Cantidad de outliers detectados:** `____________________________`
- **¿Te parece lógico ese resultado?:**  
  `______________________________________________________________________`

### 7.3.2 Cómo abordar outliers según el contexto

Antes de decidir qué hacer con un outlier, responde:

1. ¿Parece un error de captura o un caso de negocio válido?  
   `______________________________________________________________________`
2. ¿Afecta una métrica crítica, un dashboard o una segmentación?  
   `______________________________________________________________________`
3. ¿Conviene corregir, eliminar, recortar, transformar o separar?  
   `______________________________________________________________________`

**Estrategias posibles**
- Validar y corregir
- Eliminar
- Capping / winsorization
- Transformación
- Segmentación

In [ ]:

# 7.3.2 A) Separar reembolsos del gasto total
# TODO:
# - crea spend_positive
# - crea refund_amount
# - crea net_spend

tx_pos = df_transactions[df_transactions["amount_net"] > 0].copy()
tx_neg = df_transactions[df_transactions["amount_net"] < 0].copy()

spend_pos = tx_pos.groupby("customer_id")["amount_net"].sum().rename("spend_positive")
spend_neg = tx_neg.groupby("customer_id")["amount_net"].sum().rename("refund_amount")

df_customer_metrics = df_customer_metrics.merge(spend_pos, on="customer_id", how="left")
df_customer_metrics = df_customer_metrics.merge(spend_neg, on="customer_id", how="left")

df_customer_metrics["spend_positive"] = df_customer_metrics["spend_positive"].fillna(0.0)
df_customer_metrics["refund_amount"] = df_customer_metrics["refund_amount"].fillna(0.0)
df_customer_metrics["net_spend"] = df_customer_metrics["spend_positive"] + df_customer_metrics["refund_amount"]

df_customer_metrics[["customer_id", "spend_positive", "refund_amount", "net_spend", "refund_rate"]].head()

In [ ]:

# 7.3.2 B) Capping usando percentil 99
# TODO:
# - calcula el percentil 99 de net_spend
# - crea una columna recortada

metric = "net_spend"
p99 = df_customer_metrics[metric].quantile(0.99)
df_customer_metrics["net_spend_capped_p99"] = df_customer_metrics[metric].clip(upper=p99)

print(f"Percentil 99 de {metric}: {p99:,.2f}")
df_customer_metrics[[metric, "net_spend_capped_p99"]].describe().T

In [ ]:

# 7.3.2 C) Transformación logarítmica
# TODO:
# - usa una métrica no negativa
# - crea una transformación log1p
# - compara antes y después

df_customer_metrics["log_spend_positive"] = np.log1p(df_customer_metrics["spend_positive"])
df_customer_metrics[["spend_positive", "log_spend_positive"]].describe().T

### Pregunta clave

Si vas a construir un segmento **VIP**, ¿qué variable usarías?

- [ ] `total_spend`
- [ ] `spend_positive`
- [ ] `net_spend`
- [ ] `net_spend_capped_p99`

**Justifica tu elección:**

`______________________________________________________________________`

`______________________________________________________________________`

### 7.3.3 Segmentación de clientes con sentencias `if`

Una segmentación sencilla puede basarse en:

- **Recency:** días desde la última compra
- **Monetary:** gasto del cliente

Antes de programar, define tus reglas:

- **VIP** = `____________________________________________________________`
- **Leal** = `____________________________________________________________`
- **En riesgo** = `____________________________________________________________`
- **Nuevo / Inactivo** = `____________________________________________________________`

In [ ]:

# 7.3.3 A) Segmentación con if/elif/else
# Revisa las reglas y modifícalas si quieres experimentar.

def segment_customer(recency_days: int, net_spend: float) -> str:
    if (net_spend >= 2000) and (recency_days <= 30):
        return "VIP"
    elif (net_spend >= 800) and (recency_days <= 60):
        return "Leal"
    elif (net_spend >= 800) and (recency_days > 60):
        return "En riesgo"
    else:
        return "Nuevo / Inactivo"

df_customer_metrics["segment_if"] = df_customer_metrics.apply(
    lambda row: segment_customer(row["recency_days"], row["net_spend"]),
    axis=1
)

df_customer_metrics["segment_if"].value_counts(dropna=False)

In [ ]:

# TODO:
# Muestra ejemplos de clientes por segmento.
# Luego responde:
# - ¿los segmentos parecen balanceados?
# - ¿qué umbral ajustarías?

(df_customer_metrics[["customer_id", "plan", "recency_days", "net_spend", "segment_if"]]
 .sort_values(["segment_if", "net_spend"], ascending=[True, False])
 .head(15))

### Mini-ejercicio de segmentación

Ajusta al menos **un umbral** y registra qué cambió:

- **Umbral modificado:** `______________________________________`
- **Qué cambió en el tamaño de los segmentos:**  
  `______________________________________________________________________`
- **Qué segmento te parece más sensible a pequeños cambios:**  
  `______________________________________________________________________`

### 7.3.4 Segmentación usando `apply()` y múltiples columnas

A veces no basta con una sola etiqueta. También puede ser útil devolver:

- `segment_v2`
- `risk_flag`
- `recommended_action`

Piensa antes de programar:

- ¿Qué caso especial amerita una regla aparte?  
  `______________________________________________________________________`
- ¿Qué acción sugerirías para un cliente con alto gasto pero alta tasa de reembolsos?  
  `______________________________________________________________________`

In [ ]:

# 7.3.4 A) Función que retorna múltiples columnas
# TODO:
# revisa la lógica y ajusta reglas si quieres probar otros escenarios.

def segment_and_action(recency_days: int, net_spend: float, refund_rate: float) -> pd.Series:
    if refund_rate >= 0.20 and net_spend > 0:
        return pd.Series({
            "segment_v2": "Reembolsos altos",
            "risk_flag": "ALTO",
            "recommended_action": "Revisar políticas / fraude / experiencia"
        })

    if (net_spend >= 2000) and (recency_days <= 30):
        return pd.Series({
            "segment_v2": "VIP",
            "risk_flag": "BAJO",
            "recommended_action": "Upsell + atención prioritaria"
        })
    elif (net_spend >= 800) and (recency_days <= 60):
        return pd.Series({
            "segment_v2": "Leal",
            "risk_flag": "BAJO",
            "recommended_action": "Cross-sell + fidelización"
        })
    elif (net_spend >= 800) and (recency_days > 60):
        return pd.Series({
            "segment_v2": "En riesgo",
            "risk_flag": "MEDIO",
            "recommended_action": "Campaña de reactivación"
        })
    else:
        if recency_days <= 30:
            return pd.Series({
                "segment_v2": "Nuevo",
                "risk_flag": "MEDIO",
                "recommended_action": "Onboarding / activación"
            })
        return pd.Series({
            "segment_v2": "Inactivo",
            "risk_flag": "ALTO",
            "recommended_action": "Winback o limpieza de CRM"
        })

new_cols = df_customer_metrics.apply(
    lambda row: segment_and_action(row["recency_days"], row["net_spend"], row["refund_rate"]),
    axis=1
)

df_customer_metrics = pd.concat([df_customer_metrics, new_cols], axis=1)

df_customer_metrics[["segment_v2", "risk_flag", "recommended_action"]].value_counts().head(10)

In [ ]:

# TODO:
# Construye una tabla de distribución por segmento_v2.
# Agrega porcentaje y comenta cuál es el segmento dominante.

seg_dist = (
    df_customer_metrics["segment_v2"]
    .value_counts()
    .rename_axis("segment_v2")
    .reset_index(name="count")
)
seg_dist["pct"] = seg_dist["count"] / seg_dist["count"].sum()
seg_dist

### Registro de la segmentación avanzada

Completa:

- **Segmento con más clientes:** `____________________________`
- **Segmento con menor tamaño:** `____________________________`
- **Cantidad de clientes con `risk_flag = ALTO`:** `____________________________`
- **Acción de negocio que te parece más importante:**  
  `______________________________________________________________________`
- **¿Qué segmento atenderías primero y por qué?:**  
  `______________________________________________________________________`

### Nota técnica

`apply(axis=1)` es muy claro para aprender, pero en datasets grandes puede ser más lento.

Completa:

- Una alternativa más vectorizada sería `_____________________________________`
- La ventaja principal de una solución vectorizada es  
  `______________________________________________________________________`

### 7.3.5 Redacción de un Statistical Summary

Un Statistical Summary debería responder:

1. ¿Cuál era el objetivo del análisis?
2. ¿Qué datos se usaron?
3. ¿Qué métodos se aplicaron?
4. ¿Qué hallazgos relevantes aparecieron?
5. ¿Qué limitaciones existen?
6. ¿Qué recomendación de negocio harías?

In [ ]:

# 7.3.5 A) KPIs de contexto para el summary
# Ejecuta esta celda y usa los resultados para redactar tu resumen.

n = len(df_customer_metrics)
out_iqr = df_customer_metrics["outlier_iqr_total_spend"].sum()
out_iqr_pct = out_iqr / n

seg_counts = df_customer_metrics["segment_v2"].value_counts()
vip_count = seg_counts.get("VIP", 0)
risk_high = (df_customer_metrics["risk_flag"] == "ALTO").sum()

spend_desc = df_customer_metrics["spend_positive"].describe()

print(f"Clientes: {n:,}")
print(f"Outliers IQR (total_spend): {out_iqr:,} ({out_iqr_pct:.2%})")
print(f"VIP (segment_v2): {vip_count:,} ({vip_count/n:.2%})")
print(f"Riesgo ALTO: {risk_high:,} ({risk_high/n:.2%})")
print("\nDescriptivos spend_positive:")
display(spend_desc)

### Plantilla de Statistical Summary

Completa con tus resultados:

**Objetivo del análisis**  
`______________________________________________________________________`

**Datos utilizados**  
`______________________________________________________________________`

**Métodos aplicados (IQR, Z-score, segmentación, etc.)**  
`______________________________________________________________________`

**Hallazgos clave**  
- `______________________________________________________________`
- `______________________________________________________________`
- `______________________________________________________________`

**Limitaciones**  
`______________________________________________________________________`

**Recomendación para stakeholders**  
`______________________________________________________________________`

## 7.4 · Almacenando y compartiendo análisis

En analítica profesional no basta con “tener un notebook que funciona”. También necesitas:

- reproducibilidad,
- colaboración,
- trazabilidad,
- orden del proyecto.

### Completa con tus palabras

- **Repositorio:** `________________________________________________________`
- **Commit:** `____________________________________________________________`
- **Branch:** `____________________________________________________________`
- **Pull Request:** `______________________________________________________`
- **README:** `____________________________________________________________`

### 7.4.2 Estructura simple de un proyecto analítico

Observa esta estructura y anota qué guardarías en cada carpeta:

```text
mi-proyecto-analitica/
├─ data/
│  ├─ raw/
│  └─ processed/
├─ notebooks/
├─ src/
├─ reports/
├─ README.md
└─ requirements.txt
```

| Elemento | ¿Qué guardarías allí? |
|---|---|
| `data/raw/` | `_________________________________________` |
| `data/processed/` | `___________________________________` |
| `notebooks/` | `________________________________________` |
| `src/` | `______________________________________________` |
| `reports/` | `__________________________________________` |
| `README.md` | `________________________________________` |

In [ ]:

# Demo opcional: crear estructura de proyecto local
# Este bloque no sube nada a GitHub; solo crea carpetas y archivos base.

import os

project_root = "mi-proyecto-analitica"
folders = [
    f"{project_root}/data/raw",
    f"{project_root}/data/processed",
    f"{project_root}/notebooks",
    f"{project_root}/src",
    f"{project_root}/reports"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

readme_path = f"{project_root}/README.md"
gitignore_path = f"{project_root}/.gitignore"
requirements_path = f"{project_root}/requirements.txt"

if not os.path.exists(readme_path):
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write("# Mi proyecto de analítica\n\nDescripción corta del objetivo del análisis.\n")

if not os.path.exists(gitignore_path):
    with open(gitignore_path, "w", encoding="utf-8") as f:
        f.write("data/raw/\ndata/processed/\n__pycache__/\n.ipynb_checkpoints/\n.env\n")

if not os.path.exists(requirements_path):
    with open(requirements_path, "w", encoding="utf-8") as f:
        f.write("pandas\nnumpy\nmatplotlib\n")

print("Estructura creada en:", project_root)

### 7.4.3 Aplicando GitHub Workflow con Google Colab

Completa el flujo general:

1. Crear repositorio en GitHub.
2. Clonar el repo en Colab o en tu entorno local.
3. Crear una rama de trabajo.
4. Hacer cambios en notebooks / scripts / README.
5. Registrar cambios con commit.
6. Subir la rama.
7. Abrir Pull Request.

### Mini-glosario Git/GitHub

| Término | ¿Qué significa con tus palabras? |
|---|---|
| Repo | `_______________________________________________` |
| Commit | `_____________________________________________` |
| Branch | `_____________________________________________` |
| PR | `_________________________________________________` |
| Issue | `______________________________________________` |

## 6. Take aways de la sesión teórica

Completa las ideas clave con tus palabras:

1. Un outlier no siempre significa  
   `______________________________________________________________________`
2. IQR y Z-score se diferencian en que  
   `______________________________________________________________________`
3. Antes de tratar outliers, siempre conviene  
   `______________________________________________________________________`
4. La segmentación ayuda al negocio porque  
   `______________________________________________________________________`
5. Git/GitHub es importante en analítica porque  
   `______________________________________________________________________`

## 7. Cierre y próximos pasos

### Antes de la siguiente clase, marca lo que debes repasar

- [ ] `describe()`, `quantile()`, `mean()`, `std()`
- [ ] Regla de IQR
- [ ] Interpretación de Z-score
- [ ] `apply(axis=1)`
- [ ] `groupby()`
- [ ] Conceptos básicos de Git/GitHub

### Dudas pendientes de esta sesión

- `______________________________________________________________`
- `______________________________________________________________`
- `______________________________________________________________`

## 8. Información complementaria y recursos

- Documentación de `pandas`: `describe`, `quantile`, `groupby`, `apply`.
- Documentación de `matplotlib`: histogramas y boxplots.
- Git Book.
- GitHub Docs.
- Guías de trabajo con Git desde Google Colab.

### Recursos que quiero revisar después

- `______________________________________________________________`
- `______________________________________________________________`

## Cierre

### Kahoot / autoevaluación rápida

Responde sin mirar la explicación:

1. ¿Qué ventaja tiene IQR frente a Z-score en variables sesgadas?  
   `______________________________________________________________________`
2. ¿Qué diferencia hay entre `total_spend` y `net_spend`?  
   `______________________________________________________________________`
3. ¿Para qué sirve `apply(axis=1)` en este notebook?  
   `______________________________________________________________________`
4. ¿Por qué Git/GitHub mejora la trazabilidad del análisis?  
   `______________________________________________________________________`

### ¿Necesitas ayuda?
- `DATA-CO-LEARNING`: revisa los horarios de apoyo en la guía del estudiante.
- `DA_CONSULTA`: publica dudas de contenido o del proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar temas clave del sprint.
- `SESIONES 1:1`: agenda con anticipación si necesitas apoyo individual.

## Siguientes pasos

- **Próxima sesión:** práctica / proyecto aplicado del sprint.
- **Recomendación:** vuelve a ejecutar el notebook y agrega comentarios con tus propias conclusiones.
- **Meta personal para la próxima clase:**  
  `______________________________________________________________________`